<a href="https://colab.research.google.com/github/Blessy1675/Mango-Shelf-Life-MLOps/blob/main/Mango_Shelf_Life_MLOps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# STEP 1: Install required libraries

!pip install -q pandas numpy scikit-learn mlflow fastapi uvicorn pyngrok nest-asyncio joblib httpx pytest


In [3]:

# STEP 2: Import libraries

import os
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Create project folders

os.makedirs("data", exist_ok=True)
os.makedirs("features", exist_ok=True)
os.makedirs("models", exist_ok=True)

print("Libraries imported successfully")
print("Project folders created successfully")


Libraries imported successfully
Project folders created successfully


In [4]:

# STEP 3: Generate Dataset Version 1

np.random.seed(43)

n = 300

maturity = np.random.uniform(1, 5, n)
temperature = np.random.uniform(12, 32, n)
brix = np.random.uniform(8, 20, n)
bruising = np.random.uniform(0, 25, n)

transport = np.random.uniform(2, 12, n)
cooling = np.random.uniform(1, 5, n)

packaging = np.random.choice(
    ["Carton", "Plastic", "Wooden"], n
)

package_effect = np.select(
    [packaging == "Carton",
     packaging == "Plastic",
     packaging == "Wooden"],
    [2, 1, -1]
)

shelf = (
    28 - 2.2*maturity + 0.2*brix
    - 0.4*np.maximum(temperature-10, 0)
    - 0.07*transport
    - 0.12*bruising
    - 0.18*cooling
    + package_effect
    + np.random.normal(0, 1.5, n)
)

shelf = np.clip(shelf, 1, 30)

df_v1 = pd.DataFrame({
    "maturity_index": maturity,
    "pulp_temperature": temperature,
    "sweetness_brix": brix,
    "bruising_percentage": bruising,
    "transport_duration": transport,
    "cooling_delay": cooling,
    "packaging_type": packaging,
    "shelf_life_days": shelf
})

df_v1.to_csv("data/mango_v1.csv", index=False)

print("Dataset Version 1 generated successfully")
print("Total records:", len(df_v1))

display(df_v1.head())


Dataset Version 1 generated successfully
Total records: 300


,maturity_index,pulp_temperature,sweetness_brix,bruising_percentage,transport_duration,cooling_delay,packaging_type,shelf_life_days
0,1.460218,23.071789,14.450852,4.081980,4.815032,4.265449,Wooden,16.998949
1,3.436266,19.017988,15.175950,18.931667,11.695450,4.133163,Wooden,13.300396
2,1.533564,22.570481,15.554248,21.543784,8.663539,2.954337,Carton,19.940553
3,1.962358,18.568014,9.926810,6.973567,7.789882,3.528043,Wooden,20.340849
4,2.308556,13.532181,17.815994,2.191665,3.455331,2.256266,Plastic,24.039094


In [5]:

# STEP 4: Generate Dataset Version 2

np.random.seed(44)

n = 300

maturity = np.random.uniform(1, 5, n)
temperature = np.random.uniform(12, 32, n)
brix = np.random.uniform(8, 20, n)
bruising = np.random.uniform(0, 25, n)

# Increased transportation and cooling delays
transport = np.random.uniform(8, 24, n)
cooling = np.random.uniform(4, 12, n)

packaging = np.random.choice(
    ["Carton", "Plastic", "Wooden"], n
)

package_effect = np.select(
    [packaging == "Carton",
     packaging == "Plastic",
     packaging == "Wooden"],
    [2, 1, -1]
)

shelf = (
    28 - 2.2*maturity + 0.2*brix
    - 0.4*np.maximum(temperature-10, 0)
    - 0.07*transport
    - 0.12*bruising
    - 0.18*cooling
    + package_effect
    + np.random.normal(0, 1.5, n)
)

shelf = np.clip(shelf, 1, 30)

df_v2 = pd.DataFrame({
    "maturity_index": maturity,
    "pulp_temperature": temperature,
    "sweetness_brix": brix,
    "bruising_percentage": bruising,
    "transport_duration": transport,
    "cooling_delay": cooling,
    "packaging_type": packaging,
    "shelf_life_days": shelf
})

df_v2.to_csv("data/mango_v2.csv", index=False)

print("Dataset Version 2 generated successfully")
print("Total records:", len(df_v2))

display(df_v2.head())


Dataset Version 2 generated successfully
Total records: 300


,maturity_index,pulp_temperature,sweetness_brix,bruising_percentage,transport_duration,cooling_delay,packaging_type,shelf_life_days
0,4.339369,21.682782,18.277734,8.991402,10.206215,7.022256,Wooden,10.238632
1,1.419184,29.619921,12.163223,3.905862,19.505179,6.855572,Carton,18.324331
2,3.978562,28.275288,13.789994,22.306115,15.193385,7.461774,Plastic,10.905045
3,2.442003,31.451441,8.566764,7.036787,11.741597,4.580210,Carton,14.213905
4,2.437243,31.101034,9.530079,8.506790,10.177817,4.564310,Carton,16.513723


In [6]:

# STEP 5: Compare datasets

print("VERSION 1")
print("Records:", len(df_v1))
print("Average Transport:",
      round(df_v1["transport_duration"].mean(), 2))
print("Average Cooling Delay:",
      round(df_v1["cooling_delay"].mean(), 2))
print("Average Shelf Life:",
      round(df_v1["shelf_life_days"].mean(), 2))

print("\nVERSION 2")
print("Records:", len(df_v2))
print("Average Transport:",
      round(df_v2["transport_duration"].mean(), 2))
print("Average Cooling Delay:",
      round(df_v2["cooling_delay"].mean(), 2))
print("Average Shelf Life:",
      round(df_v2["shelf_life_days"].mean(), 2))


VERSION 1
Records: 300
Average Transport: 7.28
Average Cooling Delay: 3.03
Average Shelf Life: 17.48

VERSION 2
Records: 300
Average Transport: 15.8
Average Cooling Delay: 8.04
Average Shelf Life: 15.72


In [7]:

# STEP 6: Combine datasets

df = pd.concat(
    [df_v1, df_v2],
    ignore_index=True
)

# Remove duplicate and missing records
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

print("Datasets combined successfully")
print("Total records:", len(df))

display(df.head())


Datasets combined successfully
Total records: 600


,maturity_index,pulp_temperature,sweetness_brix,bruising_percentage,transport_duration,cooling_delay,packaging_type,shelf_life_days
0,1.460218,23.071789,14.450852,4.081980,4.815032,4.265449,Wooden,16.998949
1,3.436266,19.017988,15.175950,18.931667,11.695450,4.133163,Wooden,13.300396
2,1.533564,22.570481,15.554248,21.543784,8.663539,2.954337,Carton,19.940553
3,1.962358,18.568014,9.926810,6.973567,7.789882,3.528043,Wooden,20.340849
4,2.308556,13.532181,17.815994,2.191665,3.455331,2.256266,Plastic,24.039094


In [8]:

# STEP 7: Feature Engineering

df["heat_exposure_index"] = (
    np.maximum(df["pulp_temperature"] - 10, 0)
    * df["transport_duration"]
)

df["handling_damage_index"] = (
    df["bruising_percentage"]
    + 2 * df["cooling_delay"]
)

print("Feature engineering completed")

display(df[[
    "pulp_temperature",
    "transport_duration",
    "bruising_percentage",
    "cooling_delay",
    "heat_exposure_index",
    "handling_damage_index"
]].head())


Feature engineering completed


,pulp_temperature,transport_duration,bruising_percentage,cooling_delay,heat_exposure_index,handling_damage_index
0,23.071789,4.815032,4.081980,4.265449,62.941091,12.612879
1,19.017988,11.695450,18.931667,4.133163,105.469427,27.197992
2,22.570481,8.663539,21.543784,2.954337,108.904851,27.452457
3,18.568014,7.789882,6.973567,3.528043,66.743817,14.029653
4,13.532181,3.455331,2.191665,2.256266,12.204854,6.704197


In [9]:

# STEP 8: Save feature files

df.to_csv(
    "features/mango_features.csv",
    index=False
)

print("Feature file saved successfully")

print("File location:")
print("features/mango_features.csv")


Feature file saved successfully
File location:
features/mango_features.csv


In [10]:

# STEP 9: Prepare training data

X = df.drop("shelf_life_days", axis=1)

y = df["shelf_life_days"]

numeric_features = [
    "maturity_index",
    "pulp_temperature",
    "sweetness_brix",
    "bruising_percentage",
    "transport_duration",
    "cooling_delay",
    "heat_exposure_index",
    "handling_damage_index"
]

categorical_features = ["packaging_type"]

preprocessor = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training and testing completed")

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


Training and testing completed
Training records: 480
Testing records: 120


In [11]:

# STEP 10: Model training

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    )
}

results = {}

for name, algorithm in models.items():

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", algorithm)
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)

    rmse = np.sqrt(
        mean_squared_error(y_test, predictions)
    )

    r2 = r2_score(y_test, predictions)

    results[name] = {
        "model": model,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

    print("\nModel:", name)
    print("MAE:", round(mae, 3))
    print("RMSE:", round(rmse, 3))
    print("R2 Score:", round(r2, 3))



Model: Random Forest
MAE: 1.681
RMSE: 2.12
R2 Score: 0.787

Model: Gradient Boosting
MAE: 1.541
RMSE: 1.907
R2 Score: 0.828


In [12]:

# STEP 11: Select best model

best_name = min(
    results,
    key=lambda x: results[x]["MAE"]
)

best_model = results[best_name]["model"]

print("Best Model Selected:", best_name)

print("MAE:", results[best_name]["MAE"])
print("RMSE:", results[best_name]["RMSE"])
print("R2 Score:", results[best_name]["R2"])


Best Model Selected: Gradient Boosting
MAE: 1.5406597729669802
RMSE: 1.9068549430165493
R2 Score: 0.8278194369394662


In [13]:

# STEP 12: Save model

joblib.dump(
    best_model,
    "models/mango_model.pkl"
)

print("Model saved successfully")


Model saved successfully


In [14]:

# STEP 13: Configure MLflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("Mango_Shelf_Life_Prediction")

print("MLflow experiment configured successfully")


2026/11/18 10:05:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/18 10:05:15 INFO mlflow.store.db.utils: Updating database tables
2026/11/18 10:05:19 INFO mlflow.tracking.fluent: Experiment with name 'Mango_Shelf_Life_Prediction' does not exist. Creating a new experiment.


MLflow experiment configured successfully


In [17]:

# STEP 14: Register model using MLflow (FIXED)

import mlflow
import mlflow.sklearn

# Configure MLflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("Mango_Shelf_Life_Prediction")

with mlflow.start_run(run_name=best_name):

    # Log parameters
    mlflow.log_param("model_name", best_name)
    mlflow.log_param("dataset_records", len(df))
    mlflow.log_param("dataset_versions", "V1 and V2")

    # Log evaluation metrics
    mlflow.log_metric(
        "MAE", results[best_name]["MAE"]
    )

    mlflow.log_metric(
        "RMSE", results[best_name]["RMSE"]
    )

    mlflow.log_metric(
        "R2", results[best_name]["R2"]
    )

    # Register model with trusted Scikit-learn types
    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="mango_model",
        registered_model_name="MangoShelfLife",
        skops_trusted_types=[
            "sklearn.tree._tree.Tree",
            "sklearn.compose._column_transformer._RemainderColsList"
        ]
    )

print("Model registered successfully in MLflow!")
print("Selected Model:", best_name)


Model registered successfully in MLflow!
Selected Model: Gradient Boosting


Successfully registered model 'MangoShelfLife'.
Created version '1' of model 'MangoShelfLife'.


In [18]:
!mlflow ui --host 0.0.0.0 --port 5000 --backend-store-uri sqlite:///mlflow.db > /content/mlflow.log 2>&1 &

In [19]:
from google.colab import output

mlflow_url = output.eval_js(
    "google.colab.kernel.proxyPort(5000)"
)

print("MLflow Dashboard:", mlflow_url)

MLflow Dashboard: https://5000-m-s-usw1b1-1x8q775pcop9b-b.us-west1.prod.r.colab.dev


In [20]:

%%writefile app.py

import joblib
import pandas as pd

from typing import Literal
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(
    title="Mango Shelf Life Prediction"
)

# Load trained model
model = joblib.load("models/mango_model.pkl")


# Define input details
class MangoShipment(BaseModel):

    maturity_index: float = Field(ge=1, le=5)

    pulp_temperature: float = Field(ge=5, le=45)

    sweetness_brix: float = Field(ge=5, le=25)

    bruising_percentage: float = Field(ge=0, le=100)

    transport_duration: float = Field(ge=0, le=72)

    cooling_delay: float = Field(ge=0, le=48)

    packaging_type: Literal[
        "Carton", "Plastic", "Wooden"
    ]


@app.get("/")
def home():

    return {
        "message": "Mango Shelf Life API is running"
    }


@app.post("/predict")
def predict(data: MangoShipment):

    # Feature engineering
    heat = (
        max(data.pulp_temperature - 10, 0)
        * data.transport_duration
    )

    damage = (
        data.bruising_percentage
        + 2 * data.cooling_delay
    )

    # Prepare input
    input_data = pd.DataFrame([{
        "maturity_index": data.maturity_index,
        "pulp_temperature": data.pulp_temperature,
        "sweetness_brix": data.sweetness_brix,
        "bruising_percentage": data.bruising_percentage,
        "transport_duration": data.transport_duration,
        "cooling_delay": data.cooling_delay,
        "packaging_type": data.packaging_type,
        "heat_exposure_index": heat,
        "handling_damage_index": damage
    }])

    # Predict shelf life
    prediction = float(model.predict(input_data)[0])

    return {
        "estimated_shelf_life_days": round(
            max(0, prediction), 2
        ),
        "unit": "days"
    }


Writing app.py


In [21]:

# Start FastAPI server

import subprocess
import time

server = subprocess.Popen([
    "uvicorn",
    "app:app",
    "--host", "0.0.0.0",
    "--port", "8000"
])

time.sleep(3)

print("FastAPI server started successfully")


FastAPI server started successfully


In [22]:

from google.colab import output

api_url = output.eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print("FastAPI URL:", api_url)
print("Swagger Documentation:", api_url + "/docs")


FastAPI URL: https://8000-m-s-usw1b1-1x8q775pcop9b-b.us-west1.prod.r.colab.dev
Swagger Documentation: https://8000-m-s-usw1b1-1x8q775pcop9b-b.us-west1.prod.r.colab.dev/docs


In [23]:
{
  "maturity_index": 3,
  "pulp_temperature": 20,
  "sweetness_brix": 15,
  "bruising_percentage": 5,
  "transport_duration": 6,
  "cooling_delay": 2,
  "packaging_type": "Carton"
}

{'maturity_index': 3,
 'pulp_temperature': 20,
 'sweetness_brix': 15,
 'bruising_percentage': 5,
 'transport_duration': 6,
 'cooling_delay': 2,
 'packaging_type': 'Carton'}

In [25]:
import requests
import time

sample = {
    "maturity_index": 3,
    "pulp_temperature": 20,
    "sweetness_brix": 15,
    "bruising_percentage": 5,
    "transport_duration": 6,
    "cooling_delay": 2,
    "packaging_type": "Carton"
}

try:
    response = requests.post(
        api_url.rstrip("/") + "/predict",
        json=sample,
        timeout=30
    )

    print("Status Code:", response.status_code)
    print("API Response:", response.text)

    response.raise_for_status()

except requests.exceptions.ConnectionError:
    print("ERROR: FastAPI server is not running. Start the server again.")

except requests.exceptions.HTTPError:
    print("ERROR: Check the input fields and app.py prediction code.")

except requests.exceptions.RequestException as e:
    print("Request Error:", e)

Status Code: 404
API Response: 
ERROR: Check the input fields and app.py prediction code.


In [26]:

%%writefile test_app.py

from fastapi.testclient import TestClient
from app import app

client = TestClient(app)


# Test 1: Home page
def test_home():

    response = client.get("/")

    assert response.status_code == 200


# Test 2: Valid prediction
def test_prediction():

    data = {
        "maturity_index": 3,
        "pulp_temperature": 20,
        "sweetness_brix": 15,
        "bruising_percentage": 5,
        "transport_duration": 6,
        "cooling_delay": 2,
        "packaging_type": "Carton"
    }

    response = client.post("/predict", json=data)

    assert response.status_code == 200

    assert "estimated_shelf_life_days" in response.json()


# Test 3: Invalid maturity
def test_invalid_maturity():

    data = {
        "maturity_index": 10,
        "pulp_temperature": 20,
        "sweetness_brix": 15,
        "bruising_percentage": 5,
        "transport_duration": 6,
        "cooling_delay": 2,
        "packaging_type": "Carton"
    }

    response = client.post("/predict", json=data)

    assert response.status_code == 422


# Test 4: Invalid packaging
def test_invalid_packaging():

    data = {
        "maturity_index": 3,
        "pulp_temperature": 20,
        "sweetness_brix": 15,
        "bruising_percentage": 5,
        "transport_duration": 6,
        "cooling_delay": 2,
        "packaging_type": "Glass"
    }

    response = client.post("/predict", json=data)

    assert response.status_code == 422


Writing test_app.py


In [27]:

!pytest -v test_app.py


============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.14.3, langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0
collected 4 items                                                              

test_app.py::test_home PASSED                                            [ 25%]
test_app.py::test_prediction PASSED                                      [ 50%]
test_app.py::test_invalid_maturity PASSED                                [ 75%]
test_app.py::test_invalid_packaging PASSED                               [100%]

============================== 4 passed in 2.05s ===============================


In [28]:

# Create GitHub Actions folder

os.makedirs(
    ".github/workflows",
    exist_ok=True
)

print("GitHub workflow folder created")


GitHub workflow folder created


In [29]:

%%writefile .github/workflows/ci.yml

name: Mango Shelf Life MLOps

on:
  push:
    branches: [main]

  pull_request:
    branches: [main]

jobs:
  test:

    runs-on: ubuntu-latest

    steps:

      - name: Checkout code
        uses: actions/checkout@v4

      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install libraries
        run: |
          pip install -r requirements.txt

      - name: Train model
        run: |
          python pipeline.py

      - name: Run tests
        run: |
          pytest -v


Writing .github/workflows/ci.yml


In [30]:

%%writefile requirements.txt

pandas
numpy
scikit-learn
mlflow
fastapi
uvicorn
pytest
joblib
httpx


Writing requirements.txt


In [31]:

import os

for root, dirs, files in os.walk("."):

    dirs[:] = [
        d for d in dirs
        if d not in [".git", "__pycache__"]
    ]

    for file in files:

        if file.endswith((".csv", ".py", ".yml", ".txt", ".pkl")):
            print(os.path.join(root, file))


./test_app.py
./requirements.txt
./app.py
./models/mango_model.pkl
./.github/workflows/ci.yml
./features/mango_features.csv
./mlruns/1/models/m-f1a46fa79ebf475c82182cff51a72246/artifacts/requirements.txt
./data/mango_v1.csv
./data/mango_v2.csv
./sample_data/mnist_test.csv
./sample_data/california_housing_train.csv
./sample_data/california_housing_test.csv
./sample_data/mnist_train_small.csv


In [33]:
import os
import zipfile

folder = "Mango_Shelf_Life_MLOps"
zip_name = "Mango_Shelf_Life_MLOps.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, file_path)

print("ZIP file created successfully!")

ZIP file created successfully!


In [34]:

import os
import shutil

project = "Mango_Shelf_Life_MLOps"

os.makedirs(project, exist_ok=True)

for folder in ["data", "features", "models", ".github"]:
    if os.path.exists(folder):
        shutil.copytree(
            folder,
            os.path.join(project, folder),
            dirs_exist_ok=True
        )

for file in [
    "app.py",
    "test_app.py",
    "requirements.txt",
    "pipeline.py"
]:
    if os.path.exists(file):
        shutil.copy(file, project)

shutil.make_archive(
    project,
    "zip",
    root_dir=".",
    base_dir=project
)

print("Project ZIP created successfully")


Project ZIP created successfully


In [35]:

from google.colab import files

files.download("Mango_Shelf_Life_MLOps.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:

%%writefile pipeline.py

import os
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

os.makedirs("data", exist_ok=True)
os.makedirs("features", exist_ok=True)
os.makedirs("models", exist_ok=True)


def generate_data(version):

    np.random.seed(42 + version)
    n = 300

    maturity = np.random.uniform(1, 5, n)
    temp = np.random.uniform(12, 32, n)
    brix = np.random.uniform(8, 20, n)
    bruising = np.random.uniform(0, 25, n)

    if version == 1:
        transport = np.random.uniform(2, 12, n)
        cooling = np.random.uniform(1, 5, n)
    else:
        transport = np.random.uniform(8, 24, n)
        cooling = np.random.uniform(4, 12, n)

    packaging = np.random.choice(
        ["Carton", "Plastic", "Wooden"], n
    )

    effect = np.select(
        [packaging == "Carton",
         packaging == "Plastic",
         packaging == "Wooden"],
        [2, 1, -1]
    )

    shelf = (
        28 - 2.2*maturity + 0.2*brix
        - 0.4*np.maximum(temp-10, 0)
        - 0.07*transport - 0.12*bruising
        - 0.18*cooling + effect
        + np.random.normal(0, 1.5, n)
    )

    df = pd.DataFrame({
        "maturity_index": maturity,
        "pulp_temperature": temp,
        "sweetness_brix": brix,
        "bruising_percentage": bruising,
        "transport_duration": transport,
        "cooling_delay": cooling,
        "packaging_type": packaging,
        "shelf_life_days": np.clip(shelf, 1, 30)
    })

    df.to_csv(f"data/mango_v{version}.csv", index=False)

    return df


# Generate datasets
v1 = generate_data(1)
v2 = generate_data(2)

df = pd.concat([v1, v2], ignore_index=True)

df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# Feature engineering
df["heat_exposure_index"] = (
    np.maximum(df["pulp_temperature"]-10, 0)
    * df["transport_duration"]
)

df["handling_damage_index"] = (
    df["bruising_percentage"] + 2*df["cooling_delay"]
)

df.to_csv("features/mango_features.csv", index=False)

# Training data
X = df.drop("shelf_life_days", axis=1)
y = df["shelf_life_days"]

num = [
    "maturity_index", "pulp_temperature",
    "sweetness_brix", "bruising_percentage",
    "transport_duration", "cooling_delay",
    "heat_exposure_index", "handling_damage_index"
]

preprocessor = ColumnTransformer([
    ("num", "passthrough", num),
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     ["packaging_type"])
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=100, random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

results = {}

for name, algorithm in models.items():

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", algorithm)
    ])

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results[name] = (model, mae, rmse, r2)

    print(name, "MAE:", mae, "RMSE:", rmse, "R2:", r2)

best = min(results, key=lambda x: results[x][1])

best_model = results[best][0]

# Save model
joblib.dump(best_model, "models/mango_model.pkl")

# MLflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Mango_Shelf_Life_Prediction")

with mlflow.start_run():

    mlflow.log_param("selected_model", best)
    mlflow.log_param("records", len(df))

    mlflow.log_metric("MAE", results[best][1])
    mlflow.log_metric("RMSE", results[best][2])
    mlflow.log_metric("R2", results[best][3])

    mlflow.sklearn.log_model(
        best_model,
        name="mango_model",
        registered_model_name="MangoShelfLife"
    )

print("Best Model:", best)
print("MLflow registration completed")
print("Pipeline completed successfully")


Writing pipeline.py
